# Multi-Agent

多智能体系统（Multi-agent systems）将复杂应用拆解为多个专业化智能体，协同解决问题。相比依赖单一智能体处理所有步骤，多智能体架构允许你将小型、专注的智能体组合成协调的工作流

## 何时使用多智能体系统？

- 单个智能体拥有太多工具，难以做出合理选择；
- 上下文或记忆过大，单个智能体无法有效跟踪；
- 任务需要专业化分工（例如：规划者、研究员、数学专家等）。

## 多智能体模式（Multi-agent Patterns）

| 模式 | 工作方式 | 控制流 | 典型用例 |
|------|--------|--------|----------|
| **工具调用（Tool Calling）** | 一个监督者智能体将其他智能体作为工具调用。这些“工具”智能体不直接与用户交互，仅执行任务并返回结果。 | 集中式：所有路由都通过调用智能体完成。 | 任务编排、结构化工作流。 |
| **交接（Handoffs）** | 当前智能体决定将控制权转移给另一个智能体。活跃智能体发生变化，用户可继续与新智能体直接交互。 | 去中心化：智能体可自主切换活跃角色。 | 多领域对话、专家接管场景。 |

### Build a supervisor agent

学习如何使用**监督者模式**构建个人助手：由中央监督者协调多个专业子智能体。

本教程演示：
- 为不同领域（日历、邮件）创建专用子智能体；
- 将子智能体封装为工具，实现集中编排；
- 对敏感操作添加人工审核（Human-in-the-loop）。

👉 教程：[Build a supervisor agent](https://docs.langchain.com/oss/python/langchain/supervisor) 

## 如何选择模式？

| 问题 | 工具调用 | 交接（Handoffs） |
|------|---------|------------------|
| 是否需要对工作流进行集中控制？ | ✅ 是 | ❌ 否 |
| 是否希望智能体直接与用户交互？ | ❌ 否 | ✅ 是 |
| 是否需要专家之间进行类人复杂对话？ | ❌ 有限 | ✅ 强大 |

> 💡 提示：你可以混合两种模式——使用交接切换活跃智能体，同时让每个智能体在其内部通过工具调用子智能体执行专业任务。

## 自定义智能体上下文（Customizing Agent Context）

多智能体设计的核心是**上下文工程（Context Engineering）**：决定每个智能体能看到哪些信息。

LangChain 提供细粒度控制能力，包括：
- 向每个智能体传递对话或状态的哪些部分；
- 为子智能体定制专用提示词（prompts）；
- 包含或排除中间推理过程；
- 为每个智能体自定义输入/输出格式。

> 系统质量高度依赖上下文工程。目标是确保每个智能体都能获得完成其任务所需的正确数据，无论它是作为工具还是活跃智能体。

## 工具调用（Tool Calling）

在工具调用模式中，一个智能体（“控制器”）将其他智能体视为工具，在需要时调用它们。

**流程**：
1. 控制器接收输入，并决定调用哪个工具（子智能体）；
2. 工具智能体根据控制器指令执行任务；
3. 工具智能体将结果返回给控制器；
4. 控制器决定下一步操作或结束流程。

<img src="./assets/LC_agent_tool.png" width="500">

> ⚠️ 作为工具使用的智能体通常**不应直接与用户对话**，其职责是执行任务并返回结果。若需子智能体与用户交互，请使用 **交接（Handoffs）** 模式。

### 实现示例

以下是最小示例：主智能体通过工具定义访问一个子智能体。

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent

# 创建子智能体
subagent1 = create_agent(model="...", tools=[...])

@tool(
    "subagent1_name",
    description="subagent1_description"
)
def call_subagent1(query: str):
    result = subagent1.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].content

agent = create_agent(model="...", tools=[call_subagent1])

**说明**：
- 主智能体在认为任务匹配子智能体描述时，会调用 `call_subagent1`；
- 子智能体独立运行并返回结果；
- 主智能体接收结果后继续编排流程。

### 可自定义的位置

你可以在以下几个关键点控制主智能体与子智能体之间的上下文传递：

- **子智能体名称（`name`）**：主智能体引用子智能体的方式，会影响提示词，需谨慎命名；
- **子智能体描述（`description`）**：主智能体“了解”的子智能体能力，直接影响其调用决策；
- **输入到子智能体的内容**：可自定义输入，以更好引导子智能体理解任务；
- **子智能体的输出**：可调整返回内容，控制主智能体如何解读结果（例如返回最终消息文本，或附加状态/元数据）。

### 控制子智能体的输入

有两种主要方式控制主智能体传递给子智能体的输入：

1. **修改提示词**：调整主智能体的提示或工具元数据（名称和描述），以更好地指导调用时机和方式；
2. **上下文注入**：通过代码动态注入无法在静态提示中表达的信息（如完整消息历史、先前结果、任务元数据等）。

In [ ]:
from langchain.agents import AgentState
from langchain.tools import tool, ToolRuntime

class CustomState(AgentState):
    example_state_key: str

@tool(
    "subagent1_name",
    description="subagent1_description"
)
def call_subagent1(query: str, runtime: ToolRuntime[None, CustomState]):
    # 自定义逻辑，将消息转换为适合子智能体的输入
    subagent_input = some_logic(query, runtime.state["messages"])
    result = subagent1.invoke({
        "messages": subagent_input,
        # 也可以根据需要在此处传递其他状态键.
        # 要确保在主代理和子代理的状态模式中都定义这些状态键
        "example_state_key": runtime.state["example_state_key"]
    })
    return result["messages"][-1].content

### 控制子智能体的输出

两种常见策略：

1. **修改提示词**：优化子智能体提示，明确要求其返回特定内容（避免遗漏关键结果）；
2. **自定义输出格式**：在代码中调整或丰富子智能体响应后再返回给主智能体。

> 常见问题：子智能体执行了工具调用或推理，但未将结果包含在最终消息中。需提醒它：控制器（和用户）只能看到最终输出，所有相关信息必须包含其中。

In [ ]:
from typing import Annotated
from langchain.agents import AgentState
from langchain.tools import InjectedToolCallId
from langgraph.types import Command


@tool(
    "subagent1_name",
    description="subagent1_description"
)
# 我们需要将 `tool_call_id` 传递给子代理，以便它能够使用该 ID 返回工具调用结果
def call_subagent1(
    query: str,
    tool_call_id: Annotated[str, InjectedToolCallId],
# 需要返回一个 `Command` 对象，以便包含除最终工具调用之外的其他内容
) -> Command:
    result = subagent1.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return Command(update={
        # 返回状态键
        "example_state_key": result["example_state_key"],
        "messages": [
            ToolMessage(
                content=result["messages"][-1].content,
                # 需要包含工具调用 ID，以便与正确的工具调用匹配。
                tool_call_id=tool_call_id
            )
        ]
    })

## 交接（Handoffs）

在交接模式中，智能体可以直接将控制权传递给彼此。**活跃智能体发生变化**，用户与当前拥有控制权的智能体直接交互。

**流程**：
1. 当前智能体判断需要另一智能体协助；
2. 它将控制权（及状态）移交至下一个智能体；
3. 新智能体直接与用户交互，直到决定再次交接或结束。
